# Fact extraction with Llama 3.1 8B on 5,000 documents (A100)

**What this tests.** Today the memory bank's typed records come from a rule-based extractor that keeps only some of
the facts people ask about. This notebook asks an open model, Llama 3.1 8B Instruct, to turn every passage of the
5,000-document memory bank into a list of standalone facts. It runs at load time, with no question in mind, and as
fast as one A100 allows (vLLM, batched). It then measures three things.

| Measure | How | Needs |
|---|---|---|
| **Answer facts kept** | The benchmark's answer facts are checked against the extracted facts of their gold documents, by the evidence audit's word check, and against today's rule-based records | nothing |
| **Correctness** | Numbers not in the source passage, duplicates. On samples, a judge model checks whether answer facts survive and whether extracted facts are correct | the judge: `OPENAI_API_KEY` |
| **Speed** | Output tokens per second, passages per second, and the hours this projects to for 5,000, 50,000 and 512,000 documents | nothing |

What decides the result is fixed in advance in `docs/FACT_EXTRACTION_PREREGISTRATION.md`. The last cells check those
criteria.

**Before you start**
1. Runtime > Change runtime type > **A100 GPU** (40 or 80 GB).
2. Secrets (the key icon in the left bar):
   - `OPENAI_API_KEY`, for the judge (gpt-5.4-mini, about USD 0.5–1);
   - optionally `HF_TOKEN` with access to `meta-llama/Llama-3.1-8B-Instruct`. Without it, the byte-identical `unsloth`
     mirror is used.
3. Runtime > Run all.

**What it takes.**
- Installing takes a few minutes.
- Getting the benchmark takes a few minutes (it is about 5 GB).
- Building the passages takes a few minutes.
- The extraction runs about 30,000 passages. Expect roughly 30–90 minutes on an A100, faster on 80 GB; the run prints
  its own speed as it goes.

`LIMIT_PASSAGES = 2000` gives a quick look first.

In [ ]:
#@title Parameters
import os
REPO = "https://github.com/Nikku03/integratedai"
BRANCH = "claude/epic-keller-gevn7j"
MODELS = "llama-3.1-8b"        #@param ["llama-3.1-8b", "llama-3.1-8b,llama-3.2-3b", "llama-3.2-3b"] {allow-input: true}
DOCS = 5000                    #@param {type:"integer"}
LIMIT_PASSAGES = 0             #@param {type:"integer"}
MAX_TOKENS = 768               #@param {type:"integer"}
QUANTIZATION = "none"          #@param ["none", "fp8"]
JUDGE = True                   #@param {type:"boolean"}
JUDGE_MODEL = "gpt-5.4-mini"   #@param ["gpt-5.4-mini", "gpt-5.4", "gpt-5-mini"] {allow-input: true}
JUDGE_FACTS = 300              #@param {type:"integer"}
JUDGE_LINES = 300              #@param {type:"integer"}
MAX_BATCHED_TOKENS = 0         #@param {type:"integer"}
SAVE_TO_DRIVE = False          #@param {type:"boolean"}
# MAX_BATCHED_TOKENS: 0 picks by GPU: 8192 on 40 GB (vLLM's A100 default), 16384 on 80 GB. A quick A/B on
#   LIMIT_PASSAGES = 2000 can try the other value.
# MODELS: llama-3.1-8b is the model under test. Add llama-3.2-3b to compare; each model runs in its own process, one after the other.
# DOCS 5000: the 5,000-document memory bank's documents (every gold document plus a sample stratified by source, seed 5).
# LIMIT_PASSAGES: 0 = every passage (about 30,000 for 5,000 documents). 2000 gives a quick look in a few minutes.
#   The criteria are only judged on a full run.
# MAX_TOKENS: the most one passage's fact list may use; passages that reach it are counted in the report.
# QUANTIZATION: "none" (bf16) is the pre-registered setting. "fp8" stores the weights in 8 bits: less memory, and its
#   output can differ slightly.
# JUDGE: JUDGE_MODEL checks JUDGE_FACTS sampled answer facts and JUDGE_LINES sampled fact lines per model.
#   It needs the OPENAI_API_KEY secret. The pre-registered criteria 1 and 2 need it.
ON_COLAB = os.path.isdir("/content")
WORKDIR = "/content/integratedai/cie" if ON_COLAB else os.environ.get("CIE_COLAB_WORKDIR", os.getcwd())
BENCH = "/content/EnterpriseRAG-Bench-json" if ON_COLAB else os.environ.get("CIE_ERB_ROOT", os.path.join(os.getcwd(), "EnterpriseRAG-Bench"))
# verification runs without a GPU: an OpenAI-compatible server (such as Ollama) stands in for vLLM
BACKEND = os.environ.get("EXTRACT_BACKEND", "vllm")
BASE_URL = os.environ.get("EXTRACT_BASE_URL", "")
MODELS = os.environ.get("EXTRACT_MODELS", MODELS)
DOCS = int(os.environ.get("CIE_ERB_DOCS", DOCS))
LIMIT_PASSAGES = int(os.environ.get("EXTRACT_LIMIT", LIMIT_PASSAGES))
QUESTIONS = int(os.environ.get("EXTRACT_QUESTIONS", "0"))
JUDGE = JUDGE and os.environ.get("EXTRACT_JUDGE", "1") != "0"
MODEL_LIST = [m.strip() for m in MODELS.split(",") if m.strip()]
WORK = os.path.join(WORKDIR, "eval_out", f"extract_facts_{DOCS}" + (f"_q{QUESTIONS}" if QUESTIONS else ""))
print("models:", MODEL_LIST, "| documents:", DOCS, "| passages:", LIMIT_PASSAGES or "all", "| output cap:", MAX_TOKENS,
      "| weights:", QUANTIZATION if QUANTIZATION != "none" else "bf16", "| judge:", JUDGE_MODEL if JUDGE else "off")
print("backend:", BACKEND, "| colab:", ON_COLAB, "| results:", WORK)

In [ ]:
#@title GPU check (an A100: 40 or 80 GB)
import os, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader,nounits", shell=True,
                     capture_output=True, text=True).stdout.strip().splitlines()
print(gpu[0] if gpu else "no GPU visible")
GPU_MEM_GB = int(gpu[0].split(",")[1]) / 1024 if gpu else 0  # memory.total is in MiB
DRIVER_MAJOR = int(gpu[0].split(",")[2].strip().split(".")[0]) if gpu else 0
if BACKEND == "vllm" and not gpu:
    raise SystemExit("this notebook needs a GPU: Runtime > Change runtime type > A100 GPU, then run all again")
# requests in flight: an 8B model in bf16 takes about 16 GB; the rest of the GPU holds the requests' KV cache
MAX_NUM_SEQS = 512 if GPU_MEM_GB >= 70 else 256 if GPU_MEM_GB >= 35 else 128
BATCHED_TOKENS = MAX_BATCHED_TOKENS or (16384 if GPU_MEM_GB >= 70 else 8192)
if gpu and DRIVER_MAJOR < 580:
    # vLLM 0.30's default build targets CUDA 13 (driver 580+); on an older driver, a datacenter GPU runs it in compatibility mode
    os.environ["VLLM_ENABLE_CUDA_COMPATIBILITY"] = "1"
    print(f"driver {DRIVER_MAJOR} is older than 580: CUDA compatibility mode is on")
if gpu and not any(g in gpu[0] for g in ("A100", "H100", "L40", "A6000")):
    print("note: not an A100; any GPU with 24 GB or more works, more slowly")
print(f"GPU memory {GPU_MEM_GB:.0f} GB -> up to {MAX_NUM_SEQS} passages in flight, {BATCHED_TOKENS} tokens per step")

In [ ]:
#@title Clone the repository and install vLLM (a few minutes)
import importlib, os, subprocess, sys
if ON_COLAB and not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/integratedai"], check=True)
elif ON_COLAB:
    # an existing checkout moves to the latest commit of the branch: a re-run never tests stale code
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd="/content/integratedai", check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], cwd="/content/integratedai", check=True)
print("code at commit:", subprocess.run(["git", "log", "-1", "--format=%h %ci %s"], cwd=WORKDIR, capture_output=True, text=True).stdout.strip())
if ON_COLAB:
    # vLLM brings its own torch; the package is installed without its optional extras (no second torch)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.30.0", "openai>=1.40"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", WORKDIR], check=True)
src = os.path.join(WORKDIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
os.chdir(WORKDIR)
if BACKEND == "vllm":
    # checked in a fresh process: this kernel never imports torch, so the extraction processes get a clean GPU
    r = subprocess.run([sys.executable, "-c", "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, "
                        "'| cuda', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))"], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-1500:])
    assert r.returncode == 0, ("vLLM did not import (output above). On a driver older than 580, install the CUDA 12.9 build: "
                               "pip install 'vllm==0.30.0' --extra-index-url https://wheels.vllm.ai/0.30.0/cu129 "
                               "--extra-index-url https://download.pytorch.org/whl/cu129")
import cie
print("cie", cie.__version__, "from", os.path.dirname(cie.__file__))

In [ ]:
#@title Keys from Colab secrets (never printed)
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # not on Colab, or no such secret
        return os.environ.get(name)

for name in ("HF_TOKEN", "OPENAI_API_KEY"):
    v = secret(name)
    if v:
        os.environ[name] = v
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (the unsloth mirror of the same weights is used)")
print("OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "not set")
if JUDGE and not os.environ.get("OPENAI_API_KEY"):
    JUDGE = False
    print("the judge is off: add the OPENAI_API_KEY secret and run this cell again to judge samples (criteria 1 and 2 need it)")

In [ ]:
#@title Get the benchmark: the repository's JSON records (about 5 GB, a few minutes)
import os, shutil, subprocess
sources = os.path.join(BENCH, "generated_data", "sources")

def _complete():
    """Every file the clone should hold is on disk (an interrupted clone or checkout is repaired or redone)."""
    if not os.path.isfile(os.path.join(BENCH, "questions.jsonl")) or not os.path.isdir(os.path.join(BENCH, ".git")):
        return False
    listed = subprocess.run(["git", "ls-files", "generated_data/sources"], cwd=BENCH, capture_output=True, text=True)
    want = sum(1 for f in listed.stdout.splitlines() if f.endswith(".json"))
    have = sum(1 for _, _, fs in os.walk(sources) for f in fs if f.endswith(".json"))
    print(f"benchmark checkout: {have:,} of {want:,} JSON records on disk")
    return listed.returncode == 0 and want > 1000 and have == want

ok = _complete()
if not ok and os.path.isdir(os.path.join(BENCH, ".git")):
    subprocess.run(["git", "checkout", "-f", "HEAD", "--", "."], cwd=BENCH)  # restores files an interrupted checkout left out
    ok = _complete()
if not ok:
    shutil.rmtree(BENCH, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/onyx-dot-app/EnterpriseRAG-Bench", BENCH], check=True)
    ok = _complete()
assert ok, "the benchmark clone is incomplete; run this cell again"
print("questions:", sum(1 for _ in open(os.path.join(BENCH, "questions.jsonl"))))

In [ ]:
#@title Build the documents and their passages (no model; the first run reads the whole corpus once for the index)
import json, os, subprocess, sys

def run(args, env=None):
    """Runs a step in its own process and shows its output as it comes."""
    print("$", " ".join(args), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=WORKDIR)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"step failed with exit code {p.returncode} (output above)")

ENV = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "20", "TOKENIZERS_PARALLELISM": "false"}
set_file = os.path.join(WORK, "set.json")
st = json.load(open(set_file)) if os.path.exists(set_file) else {}
if st.get("n_docs") == DOCS and st.get("root") == BENCH:  # WORK is per document count and question limit
    print("reusing the passages built earlier in this runtime")
else:
    run([sys.executable, "-u", "-m", "cie.eval.extract_facts", "build", "--root", BENCH, "--work", WORK, "--docs", str(DOCS)]
        + (["--questions", str(QUESTIONS)] if QUESTIONS else []), env=ENV)
    st = json.load(open(set_file))
print(f"{st['documents']:,} documents ({st['gold_documents']} gold), {st['passages']:,} passages ({st['passage_chars']:,} characters), "
      f"{st['records']:,} rule-based records; {st['failed']} documents failed to build")

In [ ]:
#@title Extract the facts: each model in its own process (vLLM, batched; the speed is printed as it goes)
import json, os, sys
for m in MODEL_LIST:
    args = [sys.executable, "-u", "-m", "cie.eval.extract_facts", "run", "--work", WORK, "--model", m, "--max-tokens", str(MAX_TOKENS),
            "--quantization", QUANTIZATION, "--name", m.replace(":", "-")]
    if BACKEND == "vllm":
        args += ["--max-num-seqs", str(MAX_NUM_SEQS), "--max-num-batched-tokens", str(BATCHED_TOKENS), "--gpu-mem", "0.92"]
    else:
        args += ["--backend", "openai", "--base-url", BASE_URL, "--concurrency", "1"]
    if LIMIT_PASSAGES:
        args += ["--limit", str(LIMIT_PASSAGES)]
    run(args, env=ENV)
    meta = json.load(open(os.path.join(WORK, f"facts_{m.replace(':', '-')}.run.json")))
    s = meta["summary"]
    print(f"\n{m} ({meta.get('model_repo')}): {s['passages']:,} passages in {s['generation_seconds']:,.0f} s "
          f"| {s['output_tokens_per_second']:,} output tokens/s | {s['passages_per_second']} passages/s | "
          f"prompt tokens from cache {s.get('prompt_tokens_from_cache')} | model load {s['model_load_seconds']} s\n")

In [ ]:
#@title Score: answer facts kept, numbers grounded, duplicates, size and speed (no model, about a minute)
import os, sys
from IPython.display import Markdown, display
run([sys.executable, "-u", "-m", "cie.eval.extract_facts", "score", "--work", WORK, "--root", BENCH, "--quiet"], env=ENV)
display(Markdown(open(os.path.join(WORK, "report.md")).read()))

In [ ]:
#@title Judge samples with an OpenAI model (answer facts kept; extracted facts correct)
import json, os, sys
from IPython.display import Markdown, display
if not JUDGE:
    print("the judge is off (JUDGE, or no OPENAI_API_KEY secret)")
else:
    run([sys.executable, "-u", "-m", "cie.eval.extract_facts", "judge", "--work", WORK, "--root", BENCH, "--model", JUDGE_MODEL,
         "--retention", str(JUDGE_FACTS), "--lines", str(JUDGE_LINES), "--quiet"], env=ENV)
    from cie.eval.extract_facts import judge_markdown
    display(Markdown(judge_markdown(json.load(open(os.path.join(WORK, "judge.json"))))))

In [ ]:
#@title Pre-registered criteria (docs/FACT_EXTRACTION_PREREGISTRATION.md)
import json, os
rep = json.load(open(os.path.join(WORK, "report.json")))
jd = json.load(open(os.path.join(WORK, "judge.json"))) if os.path.exists(os.path.join(WORK, "judge.json")) else None
full = all(m["passages_done"] == m["passage_count"] for m in rep["methods"].values())
if not full:
    print("note: not every passage was extracted (LIMIT_PASSAGES): the criteria are shown, but only a full run decides\n")
calls = sum(v["n"] for k in ("retention", "correctness") for v in (jd or {}).get(k, {}).values()) if jd else 0
judge_ok = jd is not None and calls and jd["judge_failed"] / (calls + jd["judge_failed"]) <= 0.05
rb = (jd or {}).get("retention", {}).get("rule-based records (today)")
for m in MODEL_LIST:
    name = m.replace(":", "-")
    mm = rep["methods"].get(name)
    if not mm:
        continue
    num_share = mm["lines_with_a_number_not_in_the_passage"] / max(1, mm["fact_lines"])
    ret = (jd or {}).get("retention", {}).get(name)
    cor = (jd or {}).get("correctness", {}).get(name)
    c1 = (ret["good_share"] >= 0.80 and ret["good_share"] - rb["good_share"] >= 0.15) if judge_ok and ret and rb else None
    c2 = (cor["bad_share"] <= 0.05 and cor["bad_ci95"][1] <= 0.08) if judge_ok and cor else None
    c3 = num_share <= 0.02
    show = lambda c: "undecided (no usable judge)" if c is None else ("met" if c else "NOT met")
    print(f"== {m}" + ("  (the model under test)" if m == "llama-3.1-8b" else "  (for information)"))
    if ret and rb:
        print(f"  1. answer facts covered {ret['good_share']:.0%} (rule-based records {rb['good_share']:.0%}): need >= 80% and +15 points -> {show(c1)}")
    else:
        print(f"  1. answer facts covered: {show(c1)}")
    if cor:
        print(f"  2. wrong or unsupported {cor['bad_share']:.1%} (95% up to {cor['bad_ci95'][1]:.1%}): need <= 5% and <= 8% -> {show(c2)}")
    else:
        print(f"  2. wrong or unsupported: {show(c2)}")
    print(f"  3. fact lines with a number not in the passage {num_share:.1%}: need <= 2% -> {show(c3)}")
    if m == "llama-3.1-8b":
        verdict = "all three met: next, load these facts as memory records (separately pre-registered)" if (c1 and c2 and c3) else (
            "undecided: the judge is needed for criteria 1 and 2" if None in (c1, c2) else "not all met: the rule-based records stay")
        print("  decision:", verdict, "" if full else "(not a full run: not a decision)")

In [ ]:
#@title Look at a few passages and their facts
import json, os, random
from cie.eval.extract_facts import load_facts, load_jsonl
passages = {p["id"]: p for p in load_jsonl(os.path.join(WORK, "passages.jsonl"))}
facts = {m: {r["id"]: r for r in load_facts(os.path.join(WORK, f"facts_{m.replace(':', '-')}.jsonl"))} for m in MODEL_LIST}
common = sorted(set.intersection(*(set(f) for f in facts.values()))) if facts else []
for pid in random.Random(1).sample(common, min(3, len(common))):
    p = passages[pid]
    print("=" * 100, f"\n{p['doc_title'][:90]}  [{p['source']}]\n{p['title']}\n{p['text'][:1200]}\n")
    for m, by in facts.items():
        print(f"--- {m}: {len(by[pid]['facts'])} facts ({by[pid]['output_tokens']} tokens, {by[pid]['finish_reason']})")
        for f in by[pid]["facts"][:25]:
            print("  -", f)
    print()

In [ ]:
#@title (optional) Keep the results in Google Drive
import os, shutil
if SAVE_TO_DRIVE and ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/cie_fact_extraction/" + os.path.basename(WORK)
    shutil.copytree(WORK, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns("index.json"))
    print("copied to", dest)
else:
    print(f"SAVE_TO_DRIVE is off. The results are in {WORK}: report.md, report.json, judge.json, facts_<model>.jsonl and their .run.json")